# Neural Networks from Scratch: MNIST Handwritten Digit Classification

Welcome to the hands-on lab!

In this notebook, we build and train a **two-layer neural network completely from scratch in NumPy** to recognize handwritten digits from the famous **MNIST dataset**.

### What we do:
1. **Obtain Dataset**: Download and parse raw MNIST byte archives into normalized NumPy arrays.
2. **Inspect Samples**: Visualize raw 28x28 grayscale digits.
3. **Partition Data**: Split into Training (50,000), Validation (10,000), and Test (10,000).
4. **Build Architecture**: 784 inputs -> 128 hidden neurons (ReLU) -> 10 output classes (Softmax).
5. **Train with Backprop**: Mini-batch Gradient Descent with Momentum.
6. **Validate & Monitor**: Track Train vs Validation loss/accuracy to diagnose generalization.
7. **Test on Unseen Data**: Measure final real-world test accuracy (~98%).
8. **Visualize & Learn**: Plot confusion matrix, predictions, and analyze misclassified tricky digits.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from src.dataset import load_mnist, get_batches
from src.model import NeuralNetwork
from src.visualize import (
    plot_sample_digits,
    plot_training_curves,
    plot_confusion_matrix,
    plot_test_predictions,
    plot_misclassified_examples,
)

print("Libraries imported successfully.")

## 1. Load and Inspect the MNIST Dataset

MNIST contains 70,000 grayscale images of digits ($0$ through $9$), each $28 \times 28$ pixels.
We normalize the pixel intensities from $[0, 255]$ down to $[0.0, 1.0]$ and flatten them to $784$-dimensional vectors.

In [ ]:
X_train, y_train, X_val, y_val, X_test, y_test = load_mnist(
    data_dir="data/mnist",
    val_size=10000,
    flatten=True,
    normalize=True,
)

print(f"Training set   : {X_train.shape[0]:,} images, {X_train.shape[1]} features each")
print(f"Validation set : {X_val.shape[0]:,} images (for monitoring overfitting)")
print(f"Test set       : {X_test.shape[0]:,} images (strictly unseen)")

### Visualizing Raw Digit Samples

In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(10, 5))
images_2d = X_train.reshape(-1, 28, 28)

for i, ax in enumerate(axes.flat):
    ax.imshow(images_2d[i], cmap="gray")
    ax.set_title(f"Digit: {y_train[i]}", fontsize=10, fontweight="bold")
    ax.axis("off")

plt.suptitle("Sample MNIST Training Digits", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 2. Initialize the Neural Network Architecture

We create a 2-layer Multi-Layer Perceptron (MLP):
- **Input**: $784$ pixel intensities.
- **Hidden Layer**: $128$ neurons with **ReLU** non-linearity (initialized with He/Kaiming normal initialization).
- **Output Layer**: $10$ class neurons with **Softmax** probability distribution.
- **Loss**: Categorical Cross-Entropy.

In [ ]:
model = NeuralNetwork(input_dim=784, hidden_dim=128, output_dim=10, seed=42)
total_params = 784 * 128 + 128 + 128 * 10 + 10
print(f"Model initialized with {total_params:,} parameters:")
print(f"  W1 shape: {model.W1.shape}, b1 shape: {model.b1.shape}")
print(f"  W2 shape: {model.W2.shape}, b2 shape: {model.b2.shape}")

## 3. Train and Validate Over Epochs

We train for $10$ epochs using Mini-batch Gradient Descent with Momentum ($\\alpha = 0.05, \\beta = 0.9$).

In [ ]:
epochs = 10
batch_size = 64
learning_rate = 0.05
momentum = 0.9

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

print(f"{'Epoch':<6} | {'Train Loss':<11} | {'Train Acc':<10} | {'Val Loss':<11} | {'Val Acc':<10}")
print("-" * 58)

for epoch in range(1, epochs + 1):
    for X_batch, y_batch in get_batches(X_train, y_train, batch_size=batch_size, shuffle=True):
        model.forward(X_batch)
        grads = model.backward(y_batch)
        model.update_parameters(grads, learning_rate=learning_rate, momentum=momentum)

    # Evaluate
    tr_loss, tr_acc = model.evaluate(X_train[:10000], y_train[:10000])
    val_loss, val_acc = model.evaluate(X_val, y_val)

    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"{epoch:<6} | {tr_loss:<11.4f} | {tr_acc*100:<9.2f}% | {val_loss:<11.4f} | {val_acc*100:<9.2f}%")

### Plotting Learning Dynamics (Loss and Accuracy Curves)

In [ ]:
epochs_range = range(1, epochs + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.plot(epochs_range, history["train_loss"], "o-", color="#2563eb", label="Train Loss")
ax1.plot(epochs_range, history["val_loss"], "s--", color="#dc2626", label="Validation Loss")
ax1.set_title("Cross-Entropy Loss vs Epochs", fontsize=13, fontweight="bold")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.grid(True, alpha=0.3)
ax1.legend()

# Accuracy
ax2.plot(epochs_range, [a * 100 for a in history["train_acc"]], "o-", color="#059669", label="Train Acc")
ax2.plot(epochs_range, [a * 100 for a in history["val_acc"]], "s--", color="#d97706", label="Val Acc")
ax2.set_title("Accuracy (%) vs Epochs", fontsize=13, fontweight="bold")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy (%)")
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

## 4. Test on Held-Out Test Set (10,000 Images)

Now we evaluate the network on the test partition it has never seen during training or tuning.

In [ ]:
test_probs = model.forward(X_test)
test_preds = np.argmax(test_probs, axis=1)
test_acc = np.mean(test_preds == y_test)
test_errors = np.sum(test_preds != y_test)

print(f"Final Test Accuracy: {test_acc * 100:.2f}%")
print(f"Total Errors: {test_errors} out of 10,000 images ({test_errors / 100:.2f}% error rate)")

### Confusion Matrix (Digit by Digit Performance)

In [ ]:
cm = np.zeros((10, 10), dtype=int)
for t, p in zip(y_test, test_preds):
    cm[t, p] += 1

plt.figure(figsize=(8, 7))
plt.imshow(cm, cmap="Blues", interpolation="nearest")
plt.colorbar(fraction=0.046, pad=0.04)
plt.xticks(range(10))
plt.yticks(range(10))
plt.xlabel("Predicted Digit", fontsize=12, fontweight="bold")
plt.ylabel("True Digit", fontsize=12, fontweight="bold")
plt.title("Confusion Matrix Heatmap (10,000 Test Images)", fontsize=13, fontweight="bold")

thresh = cm.max() / 2.0
for i in range(10):
    for j in range(10):
        color = "white" if cm[i, j] > thresh else "black"
        plt.text(j, i, format(cm[i, j], "d"), ha="center", va="center", color=color, fontsize=9)

plt.tight_layout()
plt.show()

## 5. Visualizing Sample Predictions (Green = Correct, Red = Error)

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(8, 8))
test_images_2d = X_test.reshape(-1, 28, 28)

for i, ax in enumerate(axes.flat):
    ax.imshow(test_images_2d[i], cmap="gray")
    correct = (y_test[i] == test_preds[i])
    color = "#15803d" if correct else "#b91c1c"
    ax.set_title(f"True: {y_test[i]} | Pred: {test_preds[i]}", color=color, fontsize=10, fontweight="bold")
    ax.axis("off")

plt.suptitle("Sample Predictions on Test Set", fontsize=13, y=0.98)
plt.tight_layout()
plt.show()

## 6. Error Analysis: Why Did the Network Make Mistakes?

Let us inspect the images the network got wrong and see what the model predicted with its confidence.

In [ ]:
error_indices = np.where(y_test != test_preds)[0][:12]
fig, axes = plt.subplots(3, 4, figsize=(10, 8))

for ax, idx in zip(axes.flat, error_indices):
    ax.imshow(test_images_2d[idx], cmap="gray")
    conf = test_probs[idx, test_preds[idx]] * 100
    ax.set_title(f"True: {y_test[idx]}\nPred: {test_preds[idx]} ({conf:.1f}%)", color="#b91c1c", fontsize=10, fontweight="bold")
    ax.axis("off")

plt.suptitle("Error Analysis: Tricky Handwritten Digits", fontsize=13, y=0.99)
plt.tight_layout()
plt.show()